# 03 · Model Development, Evaluation and Optimisation (Question 3)
This notebook **displays** the outputs of the Q3 scripts (it trains nothing). Use it for report screenshots.

| Q3 requirement | Section |
|---|---|
| Initial model and how it was improved; learning curves | 1–2 |
| Classical baseline, search space, number of configurations | 3 |
| Comparison table (trivial, baseline, initial ANN, final ANN) with metric, training time, latency | 4 |
| Error analysis of the final model | 5 |
| Hyperparameters (LR, batch size, depth), optimisers, failed approaches | 6 |
| Experiment log | 7 |

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
from src import config
R, FIG = config.REPORTS_DIR, config.FIGURES_DIR
pd.set_option("display.max_colwidth", 90)
def show(name):
    p = FIG / name
    display(Image(filename=str(p))) if p.exists() else print(f"(missing {name} - run the scripts first)")

## 1. All neural-network runs (validation machine id_04 only)
Primary metric = **macro TPR@FPR5** (one alarm threshold per machine type). *Stable* = mean over the best epoch and its two neighbours.

In [ ]:
pd.read_csv(R / "q3_runs_summary.csv")

In [ ]:
show("q3_all_runs.png"); show("q3_round0_vs_round1.png"); show("q3_improvement_path.png")

## 2. Learning curves: initial vs final ANN

In [ ]:
sel = json.loads((ROOT / "experiments" / "final_config.json").read_text())
print("Final config selected from:", sel["selected_from"], "| rule:", sel["rule"], "| candidates:", sel["n_candidates"])
display(pd.DataFrame(sel["top5"]))
show("q3_learning_curves.png"); show("q3_seed_variance.png")

## 3. Classical baseline search (30 configurations, selected on validation)

In [ ]:
b = pd.read_csv(ROOT / "experiments" / "baseline_search.csv")
display(b.sort_values("val_macro_TPR@FPR5", ascending=False).head(10))
display(b.groupby("family").agg(configs=("config", "count"), best_val_macro=("val_macro_TPR@FPR5", "max"), mean_fit_s=("fit_time_s", "mean")))

## 4. Final comparison on the TEST machine id_06 (run once)

In [ ]:
display(pd.read_csv(R / "q3_comparison.csv"))
show("q3_roc_test.png")

## 5. Error analysis (deployed final ANN)

In [ ]:
display(Markdown((R / "q3_error_analysis.md").read_text(encoding="utf-8")))
show("q3_score_hist.png"); show("q3_errors_loudness.png")
display(pd.read_csv(R / "q3_hardest_errors.csv"))

## 6. Optimisation study (Q3b)

In [ ]:
show("q3_lr_study.png"); show("q3_bs_study.png"); show("q3_depth_study.png"); show("q3_optimizer_study.png"); show("q3_ema_stability.png")

## 7. Experiment log

In [ ]:
pd.read_csv(ROOT / "experiments" / "experiment_log.csv")